In [1]:
import re
import string
import optuna
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import MaxAbsScaler
import lightgbm as lgb

warnings.filterwarnings("ignore")
np.random.seed(42)

In [2]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(f"Label distribution:\n{train['label'].value_counts()}")


Train shape: (10800, 4), Test shape: (973, 3)
Label distribution:
label
0.0    8730
1.0    2070
Name: count, dtype: int64


In [3]:
# ─────────────────────────────────────────────
# 2. TEXT PREPROCESSING
# ─────────────────────────────────────────────
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # remove URLs
    text = re.sub(r"http\S+|www\.\S+", " url ", text)
    # remove reddit user/subreddit mentions
    text = re.sub(r"u/\w+|r/\w+", " ", text)
    # remove punctuation except apostrophe
    text = re.sub(r"[^\w\s']", " ", text)
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def combine_fields(df: pd.DataFrame) -> pd.Series:
    title = df["title"].fillna("").apply(clean_text)
    body  = df["body"].fillna("").apply(clean_text)
    # title gets extra weight by repeating it
    return title + " " + title + " " + body

train["text"] = combine_fields(train)
test["text"]  = combine_fields(test)

In [4]:
# ─────────────────────────────────────────────
# 3. HAND-CRAFTED FEATURES
# ─────────────────────────────────────────────
DEPRESSION_WORDS = {
    "depressed", "depression", "sad", "hopeless", "worthless", "suicidal",
    "suicide", "kill", "die", "dying", "alone", "lonely", "empty", "numb",
    "hate", "hate myself", "anxious", "anxiety", "tired", "exhausted",
    "cry", "crying", "pain", "hurt", "suffering", "miserable", "failed",
    "failure", "pathetic", "useless", "help", "cant", "cannot", "dont",
    "do not", "no one", "nobody", "nothing", "never", "always", "every",
    "dark", "darkness", "nightmare", "relapse", "therapy", "therapist",
    "medication", "pills", "cutting", "self harm", "self-harm"
}

POSITIVE_WORDS = {
    "happy", "good", "great", "love", "awesome", "excited", "fun",
    "girlfriend", "boyfriend", "friend", "grateful", "thankful",
    "better", "improve", "success", "proud", "hope", "hope",
}


def handcrafted_features(df: pd.DataFrame) -> np.ndarray:
    feats = []
    for _, row in df.iterrows():
        text = row["text"]
        words = text.split()
        n = max(len(words), 1)

        dep_count  = sum(1 for w in words if w in DEPRESSION_WORDS)
        pos_count  = sum(1 for w in words if w in POSITIVE_WORDS)
        neg_ratio  = dep_count / n
        pos_ratio  = pos_count / n
        sentiment_diff = neg_ratio - pos_ratio

        char_len   = len(text)
        word_count = n
        avg_word   = char_len / n
        excl_count = text.count("!")
        quest_count = text.count("?")
        body_str = row.get("body", "") if isinstance(row.get("body", ""), str) else ""
        upper_ratio = sum(1 for c in body_str if c.isupper()) / max(len(body_str or ""), 1)
        # first person singular usage
        first_person = sum(1 for w in words if w in {"i", "me", "my", "myself", "mine"}) / n

        feats.append([
            dep_count, pos_count, neg_ratio, pos_ratio, sentiment_diff,
            char_len, word_count, avg_word,
            excl_count, quest_count, upper_ratio,
            first_person,
        ])
    return np.array(feats, dtype=np.float32)


print("Extracting hand-crafted features...")
X_hand_train = handcrafted_features(train)
X_hand_test  = handcrafted_features(test)


Extracting hand-crafted features...


In [5]:
# ─────────────────────────────────────────────
# 4. TF-IDF FEATURES
# ─────────────────────────────────────────────
print("Fitting TF-IDF vectorizers...")

# Word n-grams (1,2)
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=100_000,
    sublinear_tf=True,
    min_df=2,
    analyzer="word",
    token_pattern=r"\w{2,}",
)

# Char n-grams (3,5) - catches morphology, typos
char_tfidf = TfidfVectorizer(
    ngram_range=(3, 5),
    max_features=100_000,
    sublinear_tf=True,
    min_df=3,
    analyzer="char_wb",
)

all_text = pd.concat([train["text"], test["text"]])
word_tfidf.fit(all_text)
char_tfidf.fit(all_text)

X_word_train = word_tfidf.transform(train["text"])
X_word_test  = word_tfidf.transform(test["text"])
X_char_train = char_tfidf.transform(train["text"])
X_char_test  = char_tfidf.transform(test["text"])

# stack sparse + dense
X_train_sparse = hstack([X_word_train, X_char_train])
X_test_sparse  = hstack([X_word_test,  X_char_test])

X_hand_train_sp = csr_matrix(X_hand_train)
X_hand_test_sp  = csr_matrix(X_hand_test)

X_train = hstack([X_train_sparse, X_hand_train_sp])
X_test  = hstack([X_test_sparse,  X_hand_test_sp])

y_train = train["label"].astype(int).values
print(f"Feature matrix: {X_train.shape}")

Fitting TF-IDF vectorizers...
Feature matrix: (10800, 133763)


In [ ]:
N_SPLITS = 3
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "max_bin": trial.suggest_int("max_bin", 63, 255),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    f1_scores = []

    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(50, verbose=False)
            ],
        )

        preds = model.predict_proba(X_val)[:, 1]
        preds = (preds >= 0.5).astype(int)

        f1_scores.append(f1_score(y_val, preds))

    return np.mean(f1_scores)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best F1:", study.best_value)
print("Best params:", study.best_params)


=== LightGBM Cross-Validation ===
  Fold 1: F1 = 0.8068  (best iter: 161)
  Fold 2: F1 = 0.8043  (best iter: 150)
  Fold 3: F1 = 0.8005  (best iter: 146)
  Fold 4: F1 = 0.7850  (best iter: 131)
  Fold 5: F1 = 0.7864  (best iter: 149)

Mean CV F1: 0.7966 +/- 0.0091


In [23]:
# ─────────────────────────────────────────────
# 6. THRESHOLD TUNING FOR F1
# ─────────────────────────────────────────────
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.2, 0.8, 0.01):
    pred = (oof_preds >= t).astype(int)
    f = f1_score(y_train, pred)
    if f > best_f1:
        best_f1, best_t = f, t
 
print(f"Best OOF F1: {best_f1:.4f} at threshold {best_t:.2f}")


Best OOF F1: 0.8011 at threshold 0.43


In [24]:
# ─────────────────────────────────────────────
# 7. SUBMISSION
# ─────────────────────────────────────────────
final_labels = (test_preds >= best_t).astype(int)
 
submission = pd.DataFrame({"id": test["id"], "label": final_labels})
submission.to_csv("submission_3.csv", index=False)
print(f"\nSubmission saved -> submission.csv")
print(submission["label"].value_counts())
print(submission.head(10))



Submission saved -> submission.csv
label
0    736
1    237
Name: count, dtype: int64
   id  label
0   0      0
1   1      0
2   2      0
3   3      0
4   4      1
5   5      0
6   6      0
7   7      1
8   8      0
9   9      1
